In [1]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime
import joblib
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

def load_and_prepare_data():
    cancer = load_breast_cancer()
    X, y = cancer.data, cancer.target
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, cancer.feature_names, cancer.target_names

def plot_confusion_matrix(y_true, y_pred, target_names, model_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    return plt.gcf()

def plot_feature_importance(model, feature_names, model_name):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        plt.figure(figsize=(10, 6))
        plt.title(f'Feature Importances - {model_name}')
        plt.bar(range(len(importances)), importances[indices])
        plt.xticks(range(len(importances)), 
                   [feature_names[i] for i in indices], rotation=45, ha='right')
        plt.tight_layout()
        return plt.gcf()
    return None

def plot_predictions_distribution(y_true, y_pred, target_names, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    unique, counts = np.unique(y_true, return_counts=True)
    axes[0].bar([target_names[i] for i in unique], counts, color='skyblue')
    axes[0].set_title('Actual Distribution')
    axes[0].set_ylabel('Count')
    
    unique, counts = np.unique(y_pred, return_counts=True)
    axes[1].bar([target_names[i] for i in unique], counts, color='lightcoral')
    axes[1].set_title('Predicted Distribution')
    axes[1].set_ylabel('Count')
    
    fig.suptitle(f'Class Distribution - {model_name}')
    plt.tight_layout()
    return fig

def train_and_log_model(model, model_name, param_grid, X_train, X_test, y_train, y_test, 
                        feature_names, target_names, experiment_name):
    
    with mlflow.start_run(run_name=model_name):
        print(f"Training {model_name}")
        
        grid_search = GridSearchCV(
            model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
        )
        grid_search.fit(X_train, y_train)
        
        best_model = grid_search.best_estimator_
        
        y_pred_train = best_model.predict(X_train)
        y_pred_test = best_model.predict(X_test)
        
        metrics = {
            'train_accuracy': accuracy_score(y_train, y_pred_train),
            'test_accuracy': accuracy_score(y_test, y_pred_test),
            'test_precision': precision_score(y_test, y_pred_test, average='weighted'),
            'test_recall': recall_score(y_test, y_pred_test, average='weighted'),
            'test_f1': f1_score(y_test, y_pred_test, average='weighted')
        }
        
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_param("model_type", model_name)
        
        mlflow.log_metrics(metrics)
        
        signature = infer_signature(X_train, best_model.predict(X_train))
        
        model_info = mlflow.sklearn.log_model(
            best_model, 
            "model",
            signature=signature,
            registered_model_name=f"{model_name.replace(' ', '_')}_BreastCancer"
        )
        
        print(f"Model logged at: {model_info.model_uri}")
        
        cm_fig = plot_confusion_matrix(y_test, y_pred_test, target_names, model_name)
        mlflow.log_figure(cm_fig, f"confusion_matrix_{model_name}.png")
        plt.close(cm_fig)
        
        fi_fig = plot_feature_importance(best_model, feature_names, model_name)
        if fi_fig:
            mlflow.log_figure(fi_fig, f"feature_importance_{model_name}.png")
            plt.close(fi_fig)
        
        pred_dist_fig = plot_predictions_distribution(y_test, y_pred_test, target_names, model_name)
        mlflow.log_figure(pred_dist_fig, f"prediction_distribution_{model_name}.png")
        plt.close(pred_dist_fig)
        
        report = classification_report(y_test, y_pred_test, target_names=target_names)
        with open(f"reports/classification_report_{model_name}.txt", "w") as f:
            f.write(report)
        mlflow.log_artifact(f"reports/classification_report_{model_name}.txt")
        
        print(f"\n{model_name} Results:")
        print(f"Best Parameters: {grid_search.best_params_}")
        print(f"Test Accuracy: {metrics['test_accuracy']:.4f}")
        print(f"Test F1 Score: {metrics['test_f1']:.4f}")
        
        return metrics, grid_search.best_params_, best_model

def compare_models(results_dict):
    models = list(results_dict.keys())
    
    metrics_to_compare = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.ravel()
    
    for idx, metric in enumerate(metrics_to_compare):
        values = [results_dict[model][metric] for model in models]
        axes[idx].bar(models, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
        axes[idx].set_title(f'{metric.replace("_", " ").title()}')
        axes[idx].set_ylabel('Score')
        axes[idx].set_ylim([0, 1])
        axes[idx].tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for i, v in enumerate(values):
            axes[idx].text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    return fig

def plot_model_performance_comparison(results_dict):
    models = list(results_dict.keys())
    metrics = ['test_accuracy', 'test_precision', 'test_recall', 'test_f1']
    
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(models))
    width = 0.2
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    for i, metric in enumerate(metrics):
        values = [results_dict[model][metric] for model in models]
        ax.bar(x + i * width, values, width, label=metric.replace('test_', '').title(), 
               color=colors[i], alpha=0.8)
    
    ax.set_xlabel('Models', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Model Performance Comparison Across All Metrics', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(models, rotation=45, ha='right')
    ax.legend(loc='lower right')
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    return fig

def plot_best_model_predictions(best_model, X_test, y_test, target_names, model_name):
    y_pred = best_model.predict(X_test)
    
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', 
                xticklabels=target_names, yticklabels=target_names, ax=ax1,
                cbar_kws={'label': 'Count'})
    ax1.set_title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')
    ax1.set_ylabel('True Label', fontsize=11)
    ax1.set_xlabel('Predicted Label', fontsize=11)
    
    ax2 = fig.add_subplot(gs[0, 2])
    for i, class_name in enumerate(target_names):
        mask = y_test == i
        class_acc = accuracy_score(y_test[mask], y_pred[mask])
        ax2.bar(i, class_acc, color=['#2ecc71', '#e74c3c'][i], alpha=0.7)
        ax2.text(i, class_acc + 0.02, f'{class_acc:.3f}', ha='center', fontweight='bold')
    ax2.set_xticks(range(len(target_names)))
    ax2.set_xticklabels(target_names, rotation=45, ha='right')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim([0, 1.1])
    ax2.set_title('Per-Class Accuracy', fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    ax3 = fig.add_subplot(gs[1, 2])
    correct = np.sum(y_test == y_pred)
    incorrect = len(y_test) - correct
    ax3.pie([correct, incorrect], labels=['Correct', 'Incorrect'], 
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
    ax3.set_title(f'Prediction Accuracy\n({correct}/{len(y_test)} correct)', fontweight='bold')
    
    ax4 = fig.add_subplot(gs[2, :])
    if hasattr(best_model, 'predict_proba'):
        y_proba = best_model.predict_proba(X_test)
        confidence = np.max(y_proba, axis=1)
        correct_mask = y_test == y_pred
        
        sorted_indices = np.argsort(confidence)
        
        colors_arr = np.array(['#2ecc71' if c else '#e74c3c' for c in correct_mask[sorted_indices]])
        ax4.bar(range(len(confidence)), confidence[sorted_indices], color=colors_arr, alpha=0.6)
        ax4.axhline(y=0.5, color='black', linestyle='--', linewidth=1, alpha=0.5, label='50% threshold')
        ax4.set_xlabel('Test Samples (sorted by confidence)', fontsize=11)
        ax4.set_ylabel('Prediction Confidence', fontsize=11)
        ax4.set_title('Prediction Confidence per Sample (Green=Correct, Red=Incorrect)', 
                     fontsize=12, fontweight='bold')
        ax4.set_ylim([0, 1.05])
        ax4.legend()
        ax4.grid(axis='y', alpha=0.3)
    else:
        ax4.text(0.5, 0.5, 'Probability predictions not available for this model', 
                ha='center', va='center', fontsize=12)
        ax4.set_xlim([0, 1])
        ax4.set_ylim([0, 1])
        ax4.axis('off')
    
    fig.suptitle(f'Best Model Test Set Predictions: {model_name}', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    return fig

def plot_prediction_errors(best_model, X_test, y_test, feature_names, target_names, model_name):
    y_pred = best_model.predict(X_test)
    
    misclassified_mask = y_test != y_pred
    misclassified_indices = np.where(misclassified_mask)[0]
    
    if len(misclassified_indices) == 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.text(0.5, 0.5, 'Perfect Predictions! No errors to analyze.', 
                ha='center', va='center', fontsize=16, fontweight='bold')
        ax.axis('off')
        return fig
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax1 = axes[0, 0]
    error_by_class = [np.sum((y_test == i) & misclassified_mask) for i in range(len(target_names))]
    total_by_class = [np.sum(y_test == i) for i in range(len(target_names))]
    error_rate = [e/t if t > 0 else 0 for e, t in zip(error_by_class, total_by_class)]
    
    bars = ax1.bar(target_names, error_rate, color='#e74c3c', alpha=0.7)
    ax1.set_ylabel('Error Rate')
    ax1.set_title('Error Rate by True Class', fontweight='bold')
    ax1.set_ylim([0, max(error_rate) * 1.2 if max(error_rate) > 0 else 0.1])
    for i, (bar, rate, count) in enumerate(zip(bars, error_rate, error_by_class)):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{rate:.2%}\n({count} errors)', ha='center', va='bottom', fontsize=9)
    ax1.grid(axis='y', alpha=0.3)
    
    ax2 = axes[0, 1]
    if len(misclassified_indices) > 0:
        confusion_pairs = list(zip(y_test[misclassified_mask], y_pred[misclassified_mask]))
        confusion_text = '\n'.join([f"{target_names[true]} → {target_names[pred]}: {confusion_pairs.count((true, pred))}" 
                                   for true, pred in set(confusion_pairs)])
        ax2.text(0.1, 0.9, 'Misclassification Patterns:\n\n' + confusion_text, 
                transform=ax2.transAxes, fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax2.axis('off')
    ax2.set_title('Common Misclassification Patterns', fontweight='bold')
    
    ax3 = axes[1, 0]
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        top_indices = np.argsort(importances)[-10:][::-1]
        
        ax3.barh(range(len(top_indices)), importances[top_indices], color='#3498db', alpha=0.7)
        ax3.set_yticks(range(len(top_indices)))
        ax3.set_yticklabels([feature_names[i] for i in top_indices], fontsize=8)
        ax3.set_xlabel('Importance')
        ax3.set_title('Top 10 Features (Overall)', fontweight='bold')
        ax3.invert_yaxis()
    else:
        ax3.text(0.5, 0.5, 'Feature importance not available\nfor this model type', 
                ha='center', va='center')
        ax3.axis('off')
    
    ax4 = axes[1, 1]
    total_samples = len(y_test)
    total_errors = len(misclassified_indices)
    accuracy = 1 - (total_errors / total_samples)
    
    summary_text = f"""
    Error Analysis Summary    
    Total Test Samples: {total_samples}
    Correct Predictions: {total_samples - total_errors}
    Incorrect Predictions: {total_errors}
    
    Overall Accuracy: {accuracy:.2%}
    Overall Error Rate: {(1-accuracy):.2%}
    
    Errors by Class:
    """
    for i, name in enumerate(target_names):
        class_total = np.sum(y_test == i)
        class_errors = np.sum((y_test == i) & misclassified_mask)
        summary_text += f"\n  {name}: {class_errors}/{class_total} ({class_errors/class_total:.1%})"
    
    ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes, 
            fontsize=9, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    ax4.axis('off')
    
    fig.suptitle(f'Prediction Error Analysis - {model_name}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    return fig

def main():
    mlflow.set_tracking_uri("file:./mlruns")
    
    tracking_url_type_store = mlflow.get_tracking_uri()
    print(f"MLflow Tracking URI: {tracking_url_type_store}")
    print(f"Tracking URI Type: {'local' if tracking_url_type_store.startswith('file:') else 'remote'}")
    
    experiment_name = f"Breast_Cancer_Classification_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    mlflow.set_experiment(experiment_name)
    
    print(f"\nMLflow Experiment: {experiment_name}")
    
    X_train, X_test, y_train, y_test, feature_names, target_names = load_and_prepare_data()
    
    print(f"\nDataset Info:")
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")
    print(f"Features: {X_train.shape[1]}")
    print(f"Classes: {len(target_names)} - {target_names}")

    models_config = {
        'Random Forest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': [50, 100, 200],
                'max_depth': [5, 10, 15, None],
                'min_samples_split': [2, 5, 10]
            }
        },
        'Gradient Boosting': {
            'model': GradientBoostingClassifier(random_state=42),
            'params': {
                'n_estimators': [50, 100, 200],
                'learning_rate': [0.01, 0.1, 0.2],
                'max_depth': [3, 5, 7]
            }
        },
        'SVM': {
            'model': SVC(random_state=42),
            'params': {
                'C': [0.1, 1, 10],
                'kernel': ['rbf', 'linear'],
                'gamma': ['scale', 'auto']
            }
        },
        'Logistic Regression': {
            'model': LogisticRegression(random_state=42, max_iter=1000),
            'params': {
                'C': [0.1, 1, 10],
                'penalty': ['l2'],
                'solver': ['lbfgs', 'liblinear']
            }
        }
    }

    results = {}
    trained_models = {}
    for model_name, config in models_config.items():
        metrics, best_params, trained_model = train_and_log_model(
            config['model'], 
            model_name, 
            config['params'],
            X_train, X_test, y_train, y_test,
            feature_names, target_names, experiment_name
        )
        results[model_name] = metrics
        trained_models[model_name] = trained_model
    
    print("Creating Model Comparison Visualizations")
    
    comparison_fig = compare_models(results)
    performance_comparison_fig = plot_model_performance_comparison(results)
    
    best_model_name = max(results.items(), key=lambda x: x[1]['test_accuracy'])[0]
    best_model = trained_models[best_model_name]
    
    print(f"\nBest Model: {best_model_name}")
    print(f"Creating detailed predictions visualizations for best model...")
    
    best_model_pred_fig = plot_best_model_predictions(
        best_model, X_test, y_test, target_names, best_model_name
    )
    
    error_analysis_fig = plot_prediction_errors(
        best_model, X_test, y_test, feature_names, target_names, best_model_name
    )
        
    best_model_run_id = None
    print(f"Saving Best Model for Production: {best_model_name}")
    
    with mlflow.start_run(run_name=f"BEST_MODEL_{best_model_name.replace(' ', '_')}") as run:
        signature = infer_signature(X_test, best_model.predict(X_test))
        
        print(f"Logging model to MLflow")
        
        try:
            best_model_info = mlflow.sklearn.log_model(
                sk_model=best_model,
                artifact_path="model",
                signature=signature,
                registered_model_name=f"Best_{best_model_name.replace(' ', '_')}"
            )
            print(f"Model logged successfully to: {best_model_info.model_uri}")
        except Exception as e:
            print(f"Error logging model: {e}")
            raise
        
        mlflow.log_metrics(results[best_model_name])
        
        mlflow.log_param("model_name", best_model_name)
        mlflow.log_param("model_type", "PRODUCTION_BEST_MODEL")
        
        mlflow.log_figure(best_model_pred_fig, "predictions.png")
        mlflow.log_figure(error_analysis_fig, "error_analysis.png")
        
        best_model_run_id = run.info.run_id
        
        client = mlflow.tracking.MlflowClient()
        artifacts = client.list_artifacts(best_model_run_id)
        artifact_paths = [art.path for art in artifacts]
        
        print(f"\n Best model saved successfully!")
        print(f"  Run ID: {best_model_run_id}")
        print(f"  Artifacts saved: {artifact_paths}")
    
    with mlflow.start_run(run_name="Model_Comparison_and_Best_Model_Analysis"):
        mlflow.log_figure(comparison_fig, "model_comparison_grid.png")
        mlflow.log_figure(performance_comparison_fig, "model_performance_comparison.png")
        
        mlflow.log_figure(best_model_pred_fig, f"best_model_predictions_{best_model_name}.png")
        mlflow.log_figure(error_analysis_fig, f"best_model_error_analysis_{best_model_name}.png")
        
        mlflow.log_param("best_model", best_model_name)
        mlflow.log_metric("best_accuracy", results[best_model_name]['test_accuracy'])
        mlflow.log_metric("best_f1", results[best_model_name]['test_f1'])
        mlflow.log_metric("best_precision", results[best_model_name]['test_precision'])
        mlflow.log_metric("best_recall", results[best_model_name]['test_recall'])
        
        summary_df = pd.DataFrame(results).T
        summary_df.to_csv("reports/model_comparison_summary.csv")
        mlflow.log_artifact("reports/model_comparison_summary.csv")
        mlflow.log_param("best_model_run_id", best_model_run_id)
    
    plt.close(comparison_fig)
    plt.close(performance_comparison_fig)
    plt.close(best_model_pred_fig)
    plt.close(error_analysis_fig)
        
    summary_df = pd.DataFrame(results).T
    summary_df.to_csv("reports/model_comparison_summary.csv")
    mlflow.log_artifact("reports/model_comparison_summary.csv")
    mlflow.log_param("best_model_run_id", best_model_run_id)
    
    plt.close(comparison_fig)
    plt.close(performance_comparison_fig)
    plt.close(best_model_pred_fig)
    plt.close(error_analysis_fig)
    
    print(f"\n Best Model: {best_model_name}")
    print(f"Best Test Accuracy: {results[best_model_name]['test_accuracy']:.4f}")
    print(f"Best Test F1 Score: {results[best_model_name]['test_f1']:.4f}")
    print(f"\n All Results:")
    print(pd.DataFrame(results).T.round(4))
    
    print(f"Experiment Complete!")
    print(f"\n MLflow Tracking Info:")
    print(f"Tracking URI: {tracking_url_type_store}")
    print(f"Experiment Name: {experiment_name}")
    print(f"Best Model: {best_model_name}")
    print(f"Best Model Run ID: {best_model_run_id}")
    
    best_model_filename = f"best_model_{best_model_name.replace(' ', '_')}.pkl"
    joblib.dump(best_model, best_model_filename)
    print(f"\n Best model also saved locally as: {best_model_filename}")

    return best_model_name, experiment_name, best_model, best_model_run_id

if __name__ == "__main__":
    best_model_name, experiment_name, best_model, best_model_run_id = main()

2025/11/30 20:03:34 INFO mlflow.tracking.fluent: Experiment with name 'Breast_Cancer_Classification_20251130_200334' does not exist. Creating a new experiment.


MLflow Tracking URI: file:./mlruns
Tracking URI Type: local

MLflow Experiment: Breast_Cancer_Classification_20251130_200334

Dataset Info:
Training samples: 455
Test samples: 114
Features: 30
Classes: 2 - ['malignant' 'benign']
Training Random Forest
Fitting 5 folds for each of 36 candidates, totalling 180 fits


2025/11/30 20:03:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'Random_Forest_BreastCancer' already exists. Creating a new version of this model...
Created version '3' of model 'Random_Forest_BreastCancer'.


Model logged at: models:/m-70d901f7ce0743898a37d2fabd112fa4

Random Forest Results:
Best Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}
Test Accuracy: 0.9561
Test F1 Score: 0.9560
Training Gradient Boosting
Fitting 5 folds for each of 27 candidates, totalling 135 fits


2025/11/30 20:03:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'Gradient_Boosting_BreastCancer' already exists. Creating a new version of this model...
Created version '2' of model 'Gradient_Boosting_BreastCancer'.


...

Model logged at: models:/m-7211d255b011473eb627482d991e80c6

Logistic Regression Results:
Best Parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
Test Accuracy: 0.9737
Test F1 Score: 0.9736
Creating Model Comparison Visualizations

Best Model: SVM
Creating detailed predictions visualizations for best model...


2025/11/30 20:03:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Saving Best Model for Production: SVM
Logging model to MLflow


Registered model 'Best_SVM' already exists. Creating a new version of this model...
Created version '2' of model 'Best_SVM'.


Model logged successfully to: models:/m-88a48c2648ad4ec9a6f076dca35b5753

 Best model saved successfully!
  Run ID: 74dce705a2e0431894c9048709ec1e26
  Artifacts saved: ['error_analysis.png', 'predictions.png']

 Best Model: SVM
Best Test Accuracy: 0.9825
Best Test F1 Score: 0.9825

 All Results:
                     train_accuracy  test_accuracy  test_precision  \
Random Forest                1.0000         0.9561          0.9561   
Gradient Boosting            1.0000         0.9561          0.9569   
SVM                          0.9846         0.9825          0.9825   
Logistic Regression          0.9868         0.9737          0.9737   

                     test_recall  test_f1  
Random Forest             0.9561   0.9560  
Gradient Boosting         0.9561   0.9558  
SVM                       0.9825   0.9825  
Logistic Regression       0.9737   0.9736  
Experiment Complete!

 MLflow Tracking Info:
Tracking URI: file:./mlruns
Experiment Name: Breast_Cancer_Classification_20251130_2003

In [2]:
print("running !mlflow ui --port=5001")
print(f"open: http://localhost:5001 to view results")

running !mlflow ui --port=5001
open: http://localhost:5001 to view results


In [3]:
import mlflow
mlflow.set_tracking_uri("file:./mlruns")
runs = mlflow.search_runs(order_by=["metrics.test_accuracy DESC"])
best_run_id = runs.iloc[0]['run_id']
print(f"Best Run ID: {best_run_id}")

Best Run ID: c884777b24504671a07fdadfdb3b7085


In [4]:
#Serving bets model
# !mlflow models serve -m models:/Best_SVM/latest -p 5002 --no-conda

In [6]:
!curl -X POST http://localhost:5002/invocations \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_split": {"columns": ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness", "mean compactness", "mean concavity", "mean concave points", "mean symmetry", "mean fractal dimension", "radius error", "texture error", "perimeter error", "area error", "smoothness error", "compactness error", "concavity error", "concave points error", "symmetry error", "fractal dimension error", "worst radius", "worst texture", "worst perimeter", "worst area", "worst smoothness", "worst compactness", "worst concavity", "worst concave points", "worst symmetry", "worst fractal dimension"],"data": [[17.99, 10.38, 122.8, 1001, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871, 1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193, 25.38, 17.33, 184.6, 2019, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]]}}'

{"predictions": [0]}

In [12]:
import mlflow
import numpy as np

mlflow.set_tracking_uri("file:./mlruns")

model = mlflow.sklearn.load_model("models:/Best_SVM/latest")
_, X_test, _, _, _, _ = load_and_prepare_data()
_, X_test, _, y_test, _, _ = load_and_prepare_data()

sample_index = np.random.randint(0, len(X_test))

prediction = model.predict(X_test[sample_index].reshape(1, -1))

print(f"Sample index: {sample_index}")
print(f"Prediction: {prediction[0]}")
print(f"Actual label: {y_test[sample_index]}")

Sample index: 7
Prediction: 0
Actual label: 0
